In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
from sklearn.cluster import KMeans

%matplotlib inline

## Loading the training data

Two training sets: one with repeated gestures (same gesture done multiple times in one recording) and one with single gestures. Let me load them up and see what we're working with.

In [ ]:
# paths
repeated_dir = 'data/Repeated_gesture'
single_dir = 'data/Single_gesture'

# figure out which files belong to which gesture
# the naming is kinda inconsistent lol
gesture_prefixes = {
    'wave': 'wave',
    'inf': 'inf',       # infinity
    'eight': 'eight',
    'circle': 'circle',
    'beat3': 'beat3',
    'beat4': 'beat4'
}

def load_imu(fpath):
    raw = np.loadtxt(fpath)
    # columns: ts, Wx, Wy, Wz, Ax, Ay, Az
    # drop timestamp, keep 6D IMU
    return raw[:, 1:]

# load repeated gesture files
repeated_data = {}
for gname, prefix in gesture_prefixes.items():
    files = sorted(glob.glob(os.path.join(repeated_dir, prefix + '*.txt')))
    repeated_data[gname] = [load_imu(f) for f in files]
    print(f"{gname}: {len(files)} files, shapes: {[d.shape for d in repeated_data[gname]]}")

print()

# load single gesture files  
single_data = {}
for gname, prefix in gesture_prefixes.items():
    files = sorted(glob.glob(os.path.join(single_dir, prefix + '*.txt')))
    single_data[gname] = [load_imu(f) for f in files]
    print(f"{gname}: {len(files)} files, shapes: {[d.shape for d in single_data[gname]]}")

## Let me visualize what the data looks like

I want to see the raw IMU signals for each gesture to understand the patterns. Plotting one repeated file per gesture.

In [ ]:
channel_names = ['Wx', 'Wy', 'Wz', 'Ax', 'Ay', 'Az']

fig, axes = plt.subplots(6, 1, figsize=(14, 18))
fig.suptitle('Raw IMU signals - one repeated file per gesture', fontsize=14)

colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown']

for idx, (gname, dlist) in enumerate(repeated_data.items()):
    ax = axes[idx]
    dat = dlist[0]  # just look at the first file
    for ch in range(6):
        ax.plot(dat[:, ch], alpha=0.7, label=channel_names[ch])
    ax.set_title(gname)
    ax.legend(loc='upper right', fontsize=7)
    ax.set_xlabel('sample')

plt.tight_layout()
plt.show()

## Splitting repeated gestures into individual ones

The repeated gesture files contain the same gesture done multiple times. I need to split them into individual gestures. Let me look at the signal energy to find the "rest" periods between repetitions.

In [ ]:
# first let me look at what the energy looks like for a wave file
test_dat = repeated_data['wave'][0]
energy = np.sqrt(np.sum(test_dat**2, axis=1))

plt.figure(figsize=(14, 3))
plt.plot(energy)
plt.title('Signal energy (L2 norm) - wave01')
plt.xlabel('sample')
plt.ylabel('energy')
plt.show()
print("energy stats: min={:.3f}, max={:.3f}, mean={:.3f}".format(energy.min(), energy.max(), energy.mean()))

In [ ]:
# ok so the energy doesn't really go to zero between repetitions
# because of gravity component in accelerometer (always ~9.8)
# let me try looking at just the gyroscope channels which should be near zero when not moving

gyro_energy = np.sqrt(np.sum(test_dat[:, :3]**2, axis=1))

plt.figure(figsize=(14, 3))
plt.plot(gyro_energy)
plt.title('Gyroscope energy - wave01')
plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.5)
plt.xlabel('sample')
plt.show()
print("gyro energy stats:", gyro_energy.min(), gyro_energy.max(), gyro_energy.mean())

In [ ]:
def segment_gesture(imu_data, threshold=0.3, min_gesture_len=50, min_gap=30):
    """
    split a repeated gesture recording into individual gestures
    using gyroscope energy to find rest periods
    """
    gyro = imu_data[:, :3]
    energy = np.sqrt(np.sum(gyro**2, axis=1))
    
    # smooth it a bit
    kernel_size = 15
    kernel = np.ones(kernel_size) / kernel_size
    smooth_energy = np.convolve(energy, kernel, mode='same')
    
    # find active regions (above threshold)
    active = smooth_energy > threshold
    
    # find transitions
    segments = []
    in_gesture = False
    start = 0
    
    for i in range(len(active)):
        if active[i] and not in_gesture:
            start = i
            in_gesture = True
        elif not active[i] and in_gesture:
            if i - start >= min_gesture_len:
                segments.append((start, i))
            in_gesture = False
    
    # don't forget the last one if it ends while active
    if in_gesture and len(active) - start >= min_gesture_len:
        segments.append((start, len(active)))
    
    # merge segments that are too close together
    merged = []
    for seg in segments:
        if merged and seg[0] - merged[-1][1] < min_gap:
            merged[-1] = (merged[-1][0], seg[1])
        else:
            merged.append(seg)
    
    return [imu_data[s:e] for s, e in merged], merged

# test on wave01
segs, boundaries = segment_gesture(test_dat)
print(f"found {len(segs)} segments")
print("segment lengths:", [len(s) for s in segs])
print("boundaries:", boundaries)

In [ ]:
# visualize the segmentation for a few gestures
fig, axes = plt.subplots(3, 2, figsize=(16, 10))
fig.suptitle('Segmentation check (gyro energy + boundaries)', fontsize=13)

for idx, gname in enumerate(gesture_prefixes.keys()):
    ax = axes[idx // 2][idx % 2]
    dat = repeated_data[gname][0]
    gyro_e = np.sqrt(np.sum(dat[:, :3]**2, axis=1))
    
    # smooth for display
    k = np.ones(15) / 15
    smooth = np.convolve(gyro_e, k, mode='same')
    
    ax.plot(smooth, alpha=0.8)
    ax.set_title(gname)
    
    _, bounds = segment_gesture(dat)
    for s, e in bounds:
        ax.axvline(x=s, color='g', linestyle='--', alpha=0.6)
        ax.axvline(x=e, color='r', linestyle='--', alpha=0.6)
    
    ax.set_xlabel('sample')

plt.tight_layout()
plt.show()

In [ ]:
# hmm so the segmentation basically gives one big chunk per file...
# the gestures are done continuously without clear rest periods between reps
# each "segment" is like 1500-3000 samples but a single gesture is ~400-800 samples
# so the repeated files have maybe 3-6 reps but my method can't split them

# apply to all files to confirm
for gname in gesture_prefixes.keys():
    total_segs = 0
    for dat in repeated_data[gname]:
        segs, _ = segment_gesture(dat)
        total_segs += len(segs)
    avg_single_len = np.mean([len(s) for s in single_data[gname]])
    print(f"{gname}: {total_segs} segments from 5 files (expected ~{5 * repeated_data[gname][0].shape[0] / avg_single_len:.0f}+ individual gestures)")

## Plan B: don't split, use cyclic LR-HMM

OK so splitting the repeated gestures is not really working - the person just keeps moving without pausing between reps. The project PDF says I can use a left-to-right HMM that allows going from the last state back to the first state if I haven't split the repeated data. I think that's the way to go.

So my training data will be:
- Repeated gesture files: each file = one long sequence with the gesture repeated ~3-6 times
- Single gesture files: each file = one short sequence with one gesture

The HMM will need a cyclic left-to-right structure (last state can transition back to first state).

In [ ]:
# just use each file as one training sequence - no splitting
training_raw = {}
for gname in gesture_prefixes.keys():
    training_raw[gname] = repeated_data[gname] + single_data[gname]
    lens = [len(s) for s in training_raw[gname]]
    print(f"{gname}: {len(training_raw[gname])} sequences, lengths: {lens}")

## Vector Quantization with K-means

I need to discretize the continuous 6D IMU vectors into discrete observation labels for the HMM. Using k-means clustering. The paper recommends 50-100 clusters, let me try 70 first.

In [ ]:
# stack all training data together for k-means
all_vectors = []
for gname, seq_list in training_raw.items():
    for seq in seq_list:
        all_vectors.append(seq)

all_imu = np.vstack(all_vectors)
print("total training vectors:", all_imu.shape)
print("value ranges per channel:")
for i, ch in enumerate(channel_names):
    print(f"  {ch}: [{all_imu[:, i].min():.3f}, {all_imu[:, i].max():.3f}]")

In [ ]:
M = 70  # number of observation clusters
print(f"fitting kmeans with {M} clusters...")
kmeans = KMeans(n_clusters=M, random_state=42, n_init=10, max_iter=300)
kmeans.fit(all_imu)
print("done! inertia:", kmeans.inertia_)

# check cluster sizes - don't want empty clusters
labels_all = kmeans.labels_
cluster_sizes = np.bincount(labels_all, minlength=M)
print(f"cluster sizes: min={cluster_sizes.min()}, max={cluster_sizes.max()}, mean={cluster_sizes.mean():.1f}")
print(f"empty clusters: {np.sum(cluster_sizes == 0)}")

plt.figure(figsize=(12, 3))
plt.bar(range(M), cluster_sizes)
plt.title('K-means cluster sizes')
plt.xlabel('cluster id')
plt.ylabel('count')
plt.show()

In [ ]:
# quantize all training sequences
def quantize(seq, km):
    return km.predict(seq)

training_obs = {}  # gesture -> list of 1D int arrays
for gname, seq_list in training_raw.items():
    training_obs[gname] = [quantize(s, kmeans) for s in seq_list]
    print(f"{gname}: {len(training_obs[gname])} sequences")
    # sanity check
    print(f"  first seq labels: {training_obs[gname][0][:15]}")
    print(f"  seq lengths: {[len(s) for s in training_obs[gname]]}")

In [ ]:
import pickle

os.makedirs('models', exist_ok=True)

# save kmeans model
with open('models/kmeans_model.pkl', 'wb') as f:
    pickle.dump(kmeans, f)
print("saved kmeans model")

# save quantized training data
with open('models/training_obs.pkl', 'wb') as f:
    pickle.dump(training_obs, f)
print("saved training observations")

# also save the raw training sequences in case I need them later
with open('models/training_raw.pkl', 'wb') as f:
    pickle.dump(training_raw, f)
print("saved raw training data")

print(f"\nM = {M} clusters, total {sum(len(v) for v in training_obs.values())} training sequences")